# What Actually Goes Into the Context Window [Step 01.01]

> **MLCourse - Agentic AI - Agent Patterns**

Every LLM call is one flat string. Not a conversation, not a memory, not a
session - **one string**, assembled from scratch on every single turn and thrown
away the moment the response comes back.

Context engineering is the discipline of deciding what goes into that string.

```
        +---------------------------------------------------------+
        |                   THE CONTEXT WINDOW                    |
        +---------------------------------------------------------+
        | 1. SYSTEM PROMPT      role, rules, output format        |
        | 2. TOOL SCHEMAS       every tool's name/description/args|
        | 3. RETRIEVED DOCS     whatever RAG pulled in            |
        | 4. CONVERSATION       every prior user+assistant turn   |
        | 5. THE USER'S QUERY   usually the smallest piece        |
        +---------------------------------------------------------+
                                |
                                v
                          model generates
```

### What you'll learn

- The five components that make up essentially every agent request.
- How to **measure** each component instead of guessing at it.
- Why the local estimate and the provider's billed count differ, and which to trust.
- The single most surprising fact about real agents: **the user's question is
  almost never the expensive part.**

### Why it matters

When an agent is slow, expensive, or confused, the cause is almost always
something in components 2-4 that you never look at. You cannot fix a context
problem you have not measured, and nothing in the LangChain API shows you the
assembled string by default. This notebook makes it visible.

### Prerequisites

- [01_langchain/01_fundamentals](../../01_langchain/01_fundamentals) - chat models and messages.
- [03_rag_advanced](../../03_rag_advanced) - where "retrieved docs" come from.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. Building a realistic request by hand

We will assemble a small but *realistic* customer-support agent request. Nothing
here is special to LangChain - the same five pieces exist whether you use
LangGraph, CrewAI, or raw HTTP.

Read each piece and notice how ordinary it is. That ordinariness is the point:
context bloat is never one dramatic mistake, it is five reasonable decisions
stacked on top of each other.

### Component 1: the system prompt


In [ ]:
SYSTEM_PROMPT = """You are Ada, the support assistant for Northwind Cycles.

Rules:
- Answer only from the reference material and order records provided.
- If the answer is not there, say "I don't have that information" - never guess.
- Be concise: at most three sentences.
- Never reveal internal SKU codes or supplier names to the customer.
- Always end with a single follow-up question offering further help.
"""

# --- Component 2: tool schemas (what the model is told the tools do) ----------
TOOL_SCHEMAS = [
    {"name": "lookup_order",
     "description": "Fetch the status, items and delivery estimate for one order.",
     "parameters": {"type": "object",
                    "properties": {"order_id": {"type": "string",
                                                "description": "Order id, e.g. NW-10231"}},
                    "required": ["order_id"]}},
    {"name": "start_return",
     "description": "Open a return request for one item on an order.",
     "parameters": {"type": "object",
                    "properties": {"order_id": {"type": "string"},
                                   "sku": {"type": "string"},
                                   "reason": {"type": "string",
                                              "description": "Customer's stated reason"}},
                    "required": ["order_id", "sku", "reason"]}},
    {"name": "check_stock",
     "description": "Report how many units of a product are in stock per warehouse.",
     "parameters": {"type": "object",
                    "properties": {"sku": {"type": "string"},
                                   "warehouse": {"type": "string"}},
                    "required": ["sku"]}},
]

# --- Component 3: retrieved documents (what RAG handed us) -------------------
RETRIEVED_DOCS = [
    "Returns policy: unused items may be returned within 30 days of delivery. "
    "Items must be in original packaging. Refunds are issued to the original "
    "payment method within 5 business days of the item arriving at our warehouse.",
    "Shipping: standard delivery is 3-5 business days within the EU. Express "
    "delivery is 1-2 business days and costs an additional 9 EUR. Orders placed "
    "after 15:00 CET ship the following business day.",
    "Warranty: frames carry a 5-year warranty against manufacturing defects. "
    "Wheels and drivetrain components carry a 2-year warranty. Wear items such "
    "as tyres, brake pads and chains are not covered.",
]

# --- Component 4: the conversation so far -------------------------------------
HISTORY = [
    ("user", "Hi, I ordered a bike last week and I want to check where it is."),
    ("assistant", "Happy to help. Could you give me your order number?"),
    ("user", "It's NW-10231."),
    ("assistant", "Order NW-10231 shipped yesterday and is due Thursday. "
                  "Anything else I can check for you?"),
    ("user", "Great. One of the tyres looked flat in the photo you sent."),
    ("assistant", "Tyres can lose pressure in transit and are easy to reinflate. "
                  "Would you like me to look into a replacement anyway?"),
]

# --- Component 5: the new user query ------------------------------------------
QUERY = "Yes please - and how long do I have to send the whole bike back if I change my mind?"

print("assembled 5 components")


### 2. Measure every component

Now the actual exercise. We serialise each component the way it would be sent
and count its tokens.

> **Note on the tool schemas.** They are Python dicts here, but the provider
> serialises them to JSON and puts them *in the prompt*. There is no magic
> side-channel. A tool your agent never calls still costs you tokens on every
> single turn.

In [3]:
def render_history(turns):
    """Turn (role, text) pairs into the flat text the model actually receives."""
    return "\n".join("%s: %s" % (r.upper(), t) for r, t in turns)


def render_docs(docs):
    return "\n\n".join("[doc %d] %s" % (i + 1, d) for i, d in enumerate(docs))


components = {
    "1 system prompt":  SYSTEM_PROMPT,
    "2 tool schemas":   json.dumps(TOOL_SCHEMAS),
    "3 retrieved docs": render_docs(RETRIEVED_DOCS),
    "4 conversation":   render_history(HISTORY),
    "5 user query":     QUERY,
}

sizes = {k: approx_tokens(v) for k, v in components.items()}
total = sum(sizes.values())

print("%-18s %8s %8s  %s" % ("component", "tokens", "share", "bar"))
print("-" * 62)
for name, n in sizes.items():
    share = n / total
    print("%-18s %8d %7.1f%%  %s" % (name, n, share * 100, "#" * int(share * 40)))
print("-" * 62)
print("%-18s %8d %7.1f%%" % ("TOTAL", total, 100.0))

component            tokens    share  bar
--------------------------------------------------------------
1 system prompt          83    14.3%  #####
2 tool schemas          219    37.8%  ###############
3 retrieved docs        144    24.8%  #########
4 conversation          113    19.5%  #######
5 user query             21     3.6%  #
--------------------------------------------------------------
TOTAL                   580   100.0%


### Read that table again

The user's actual question - the only part a human would call "the input" - is a
rounding error. Everything else is scaffolding *you* chose to include.

This is the central insight of context engineering:

> You are not writing a prompt. You are **allocating a budget**, and the query is
> the smallest line item in it.

### 3. Estimate vs. reality

`approx_tokens()` uses `cl100k_base`, which is not the tokenizer Qwen uses. So
let us send the request for real and compare our estimate against the number the
provider actually billed.

Two sources of difference:

1. **Different tokenizer.** Different vocabulary, different splits.
2. **Chat template overhead.** The provider wraps every message in role markers
   and special tokens you never see. That overhead is real and you pay for it.

In [4]:
messages = [
    ("system", SYSTEM_PROMPT
     + "\n\nAvailable tools (JSON schemas):\n" + json.dumps(TOOL_SCHEMAS)
     + "\n\nReference material:\n" + render_docs(RETRIEVED_DOCS)),
] + list(HISTORY) + [("user", QUERY)]

response = chat(messages, max_tokens=160)

billed_in = response.usage_metadata["input_tokens"]
print("our local estimate :", total, "tokens")
print("provider billed    :", billed_in, "tokens")
print("difference         : %+d (%+.1f%%)" % (billed_in - total,
                                              100 * (billed_in - total) / total))
print()
print("model replied:", response.content)

our local estimate : 580 tokens
provider billed    : 654 tokens
difference         : +74 (+12.8%)

model replied: You have 30 days from delivery to return the bike if it is unused and in original packaging. Would you like me to start the return process for the tyre now?


> **Pitfall.** People budget with a local tokenizer and then get surprised by the
> bill. Use the estimate for *planning* (it is free and instant), but always
> reconcile against `usage_metadata` at least once per pipeline. The gap is
> systematic, not random - measure it once and apply it as a correction factor.

### 4. The part that grows

Four of the five components are roughly fixed per request. One is not.

Every turn appends a user message *and* an assistant message to the history, and
the whole history is resent next turn. Let us watch what that does over ten turns
with everything else held constant.

In [5]:
FIXED = sizes["1 system prompt"] + sizes["2 tool schemas"] + sizes["3 retrieved docs"]
AVG_USER = 25          # tokens in a typical user turn
AVG_ASSISTANT = 70     # tokens in a typical assistant turn

print("%5s %10s %10s %10s %12s" % ("turn", "fixed", "history", "total", "cumulative"))
print("-" * 52)
cumulative = 0
for turn in range(1, 11):
    history = turn * (AVG_USER + AVG_ASSISTANT)
    per_call = FIXED + history
    cumulative += per_call
    print("%5d %10d %10d %10d %12d" % (turn, FIXED, history, per_call, cumulative))

print()
print("turn 10 request is %.1fx the size of turn 1" %
      ((FIXED + 10 * (AVG_USER + AVG_ASSISTANT)) / (FIXED + AVG_USER + AVG_ASSISTANT)))
print("cumulative tokens over 10 turns:", cumulative)

 turn      fixed    history      total   cumulative
----------------------------------------------------
    1        446         95        541          541
    2        446        190        636         1177
    3        446        285        731         1908
    4        446        380        826         2734
    5        446        475        921         3655
    6        446        570       1016         4671
    7        446        665       1111         5782
    8        446        760       1206         6988
    9        446        855       1301         8289
   10        446        950       1396         9685

turn 10 request is 2.6x the size of turn 1
cumulative tokens over 10 turns: 9685


Two things to take from that table:

- **Per-call cost grows linearly** with turn count. Nothing about the task got
  harder; you are simply re-reading your own transcript over and over.
- **Cumulative cost grows quadratically.** A 50-turn conversation is not 5x a
  10-turn one.

This is the single most common reason a working prototype becomes an unaffordable
product, and it is why the next notebooks are about budgets and trimming rather
than about clever prompts.

### 5. What each component is actually *for*

Measuring is half the job. The other half is knowing which component a given
piece of information belongs in. Putting the right fact in the wrong component
is a real and common bug.

| Component | Holds | Lifetime | Classic mistake |
|---|---|---|---|
| System prompt | Role, rules, output format | Every turn, forever | Stuffing per-user facts in here |
| Tool schemas | What the model *can do* | Every turn | Registering 40 tools "just in case" |
| Retrieved docs | Facts too big / too fresh to memorise | This turn only | Retrieving top-20 when top-3 answers it |
| Conversation | What was already said and decided | Grows unboundedly | Never trimming it |
| User query | The immediate ask | This turn | (rarely the problem) |

Let us prove the "40 tools" point costs real money rather than just asserting it.

### What does one extra tool schema actually cost, per call, forever?


In [ ]:
one_tool = approx_tokens(json.dumps(TOOL_SCHEMAS[1]))
print("one tool schema        : %d tokens" % one_tool)
print("40 such tools          : %d tokens on EVERY call" % (one_tool * 40))
print("over a 10-turn session : %d tokens" % (one_tool * 40 * 10))
print()
print("...for tools the agent may never call once.")


### 6. Pitfalls

- **Believing the framework hides this from you.** It does not. It hides it
  *from your eyes*, not from your bill. Print the assembled messages at least
  once for every agent you build.
- **Optimising the query.** Rewriting the user's 20-token question saves 20
  tokens. Trimming the history saves hundreds.
- **Trusting local token counts as exact.** They are an estimate. Reconcile.
- **Assuming a bigger window solves it.** It shifts the failure from "request
  rejected" to "model ignores the middle of the input" - which is exactly what
  notebook 04 measures.

### Recap

| Idea | Takeaway |
|---|---|
| Five components | System, tools, docs, history, query - every agent request |
| Measure, don't guess | The query is the smallest line item, almost always |
| Estimate vs billed | Local tokenizer is approximate; reconcile with `usage_metadata` |
| History grows | Linear per call, quadratic cumulatively |
| Tools cost per turn | An unused tool schema is a permanent tax |

**Next:** [02_token_budget](02_token_budget.ipynb) turns these measurements into
an explicit budget with an enforced ceiling.